# scRareBench | High-Level scDisInFact | Dataset 5 — NYGC / Seurat v4 PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

**NYGC / Seurat v4 CITE-seq PBMC** · Google Colab (GPU / High-RAM)

`load_dataset(5) → MethodSpec(scDisInFact) → benchmark_method() → scIB + rare-cell evaluation → multi-seed HTML/ZIP`

The scRareBench dataset loader handles the official PBMC QC, counts/source contract, and registered rare scenarios. The notebook contains **no duplicated dataset-validation code**.

scDisInFact expects a condition covariate. Dataset 5 has no independent, validated experimental-condition key in the scRareBench contract; therefore we supply a single **method-only constant condition** (`baseline`) and evaluate just the **shared `mu_c` latent**. We never pass biological cell-type labels or rare-scenario annotations to the method. This is a **batch-integration baseline**, not a multi-condition disentanglement analysis.

**Dataset contract:** 152,145 post-QC cells, 20,729 genes; raw RNA counts in `adata.X`; `orig.ident` identifies 24 batches. Method-side HVGs use raw counts, Seurat-v3 and batch-aware ranking. **scIB reference uses its separate 4,000-HVG policy** as configured by scRareBench.

Model hyperparameters follow the provided working Dataset 0 scDisInFact notebook (2,000 HVGs; `Ks=[8,4]`; 100 epochs). The shared 8D latent is the representation being benchmarked. To limit GPU inference allocations, latent extraction is chunked; this does not change training or the returned representation.

## 1. Install the pinned runtime

In [ ]:
import os
import subprocess
import sys

# scIB PCR comparison uses JAX; keep its linear algebra on CPU.
# Set before importing packages that could initialize JAX.
os.environ["JAX_PLATFORMS"] = "cpu"

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"  # v0.10.6
SCDISINFACT_COMMIT = "9466ecf257d923f879cb39712c530d24826d88bc"
SCDISINFACT_GITHUB = (
    "git+https://github.com/ZhangLabGT/scDisInFact.git@" + SCDISINFACT_COMMIT
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    f"git+https://github.com/amirhossein-alishahi/scRareBench_.git@{SCRAREBENCH_REVISION}",
])

METHOD_DEPENDENCIES = (
    "torch>=1.11.0",
    "scikit-learn>=0.22.1",
    "matplotlib>=3.5.2",
    "scikit-misc>=0.3",
)

from scrarebench.runtime import setup_runtime

setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("torch", "sklearn", "matplotlib", "skmisc"),
)

# Keep the older scDisInFact dependency pins from replacing Colab's runtime.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCDISINFACT_GITHUB,
])

## 2. Load Dataset 5

In [ ]:
from scrarebench import load_dataset

adata = load_dataset(5)
print(adata)

## 3. scDisInFact method configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42
OUTPUT_DIR = "/content/scrarebench_results/dataset5_scdisinfact"

METHOD_CONFIG = {
    "batch_key": "orig.ident",
    "n_hvg": 2000,
    "hvg_span": 0.3,
    "condition_column": "_scdisinfact_condition",
    "condition_value": "baseline",
    "Ks": [8, 4],
    "batch_size": 64,
    "inference_batch_size": 1024,
    "interval": 10,
    "lr": 5e-4,
    "reg_mmd_comm": 1e-4,
    "reg_mmd_diff": 1e-4,
    "reg_gl": 1.0,
    "reg_class": 1.0,
    "reg_kl_comm": 1e-5,
    "reg_kl_diff": 1e-2,
    "enc_injection": False,
    "nepochs": 100,
    "recon_loss": "NB",
}

## 4. scDisInFact method runner

The method-only preprocessing selects raw-count HVGs, supplies `orig.ident` and one constant condition, trains scDisInFact, and restores the original cell order (the method internally rearranges cells by batch/condition). The scRareBench evaluation, rare-cell annotations, clustering and report generation are **not** reimplemented here.

In [ ]:
import gc
import random

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch

from scDisInFact import create_scdisinfact_dataset, scdisinfact
from scrarebench import MethodOutput


def run_scdisinfact(method_adata, seed, config):
    batch_key = config["batch_key"]
    barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)

    hvg = sc.pp.highly_variable_genes(
        method_adata,
        layer=None,  # Dataset 5: canonical raw counts in X
        flavor="seurat_v3",
        n_top_genes=config["n_hvg"],
        batch_key=batch_key,
        span=config["hvg_span"],
        inplace=False,
    )
    mask = hvg["highly_variable"].to_numpy(dtype=bool)
    counts = method_adata.X[:, mask]
    counts = (
        counts.tocsr().astype(np.float32, copy=False)
        if sp.issparse(counts)
        else np.asarray(counts, dtype=np.float32)
    )

    # Only method-owned covariates. No cell-type or rare-scenario labels.
    meta = pd.DataFrame(index=method_adata.obs_names.copy())
    meta[batch_key] = method_adata.obs[batch_key].astype(str).to_numpy()
    meta[config["condition_column"]] = config["condition_value"]
    meta["_barcode"] = barcodes

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    data_dict = create_scdisinfact_dataset(
        counts,
        meta,
        condition_key=[config["condition_column"]],
        batch_key=batch_key,
        batch_cond_key=None,
        log_trans=True,
    )
    model = scdisinfact(
        data_dict=data_dict,
        reg_mmd_comm=config["reg_mmd_comm"],
        reg_mmd_diff=config["reg_mmd_diff"],
        reg_gl=config["reg_gl"],
        reg_class=config["reg_class"],
        reg_kl_comm=config["reg_kl_comm"],
        reg_kl_diff=config["reg_kl_diff"],
        Ks=list(config["Ks"]),
        batch_size=config["batch_size"],
        interval=config["interval"],
        lr=config["lr"],
        seed=int(seed),
        enc_injection=config["enc_injection"],
        device=device,
    )
    model.train_model(nepochs=config["nepochs"], recon_loss=config["recon_loss"])
    model.eval()

    latent_chunks, barcode_groups = [], []
    step = int(config["inference_batch_size"])
    with torch.no_grad():
        for part, part_meta in zip(data_dict["datasets"], data_dict["meta_cells"]):
            barcode_groups.append(part_meta["_barcode"].astype(str).to_numpy())
            n_cells = part.counts_norm.shape[0]
            for start in range(0, n_cells, step):
                end = min(start + step, n_cells)
                output = model.inference(
                    counts=part.counts_norm[start:end].to(device),
                    batch_ids=part.batch_id[start:end, None].to(device),
                    print_stat=False,
                )
                latent_chunks.append(
                    output["mu_c"].detach().cpu().numpy().astype(np.float32, copy=False)
                )

    native_latent = np.concatenate(latent_chunks, axis=0)
    native_barcodes = np.concatenate(barcode_groups, axis=0)
    latent = native_latent[pd.Index(native_barcodes).get_indexer(barcodes)]

    del model, data_dict, counts, native_latent, latent_chunks
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_scdisinfact_shared",
    )

## 5. Run the high-level scRareBench benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDisInFact",
    runner=run_scdisinfact,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES + (SCDISINFACT_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=False,
    overwrite=True,
)

## 6. Results and download

In [ ]:
from IPython.display import display
from google.colab import files
from pathlib import Path
import shutil

display(result.summary().round(5))
print("Multi-seed HTML:", result.report_path)
print("Final ZIP:", result.archive_path)

archive = Path("/content/scDisInFact_dataset5_results.zip")
shutil.copy2(result.archive_path, archive)
files.download(str(archive))